<a href="https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest Regressor, predicting the same target as Week 4 (gsc_avg_position), using the same 5 honest features.

Why it fits this lane: Your baseline used Linear Regression, which can only combine features additively — it can't capture interactions (e.g., "impressions matter more when engagement is also high"). Random Forest can model those nonlinear interactions without assuming a fixed relationship shape, which matters for ranking signals since real search behavior is rarely linear. It also gives you permutation importance for free, which Section 4 needs to say which signals the model actually leans on — directly answering the "Ranking Signal Analysis" lane's core question.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 code cell
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_avg_position, gsc_impressions, ga4_engaged_sessions,
           sessions_organic, sessions_ai, scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

print("Rows before dropna:", len(df))
df = df.dropna()
print("Rows after dropna:", len(df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows before dropna: 3611061
Rows after dropna: 2082695


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_hash_id, not a plain random row split. Your Week 4 baseline used an ungrouped train_test_split — worth naming honestly as something this week fixes, not repeats. Content items from the same client likely share site-level SEO factors (domain authority, template structure, internal linking), so a random row split risks leaking a client's pattern between train and test. A grouped split ensures each client appears in only one side, giving an honest read on whether the model generalizes to new clients, not just new rows from clients it's already seen.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 code cell
from sklearn.model_selection import GroupShuffleSplit

features = ['gsc_impressions', 'ga4_engaged_sessions', 'sessions_organic', 'sessions_ai', 'scroll_events']
X = df[features]
y = df['gsc_avg_position']
groups = df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train clients:", df.iloc[train_idx]['client_hash_id'].nunique())
print("Test clients:", df.iloc[test_idx]['client_hash_id'].nunique())
print("Overlap check (should be 0):", len(set(df.iloc[train_idx]['client_hash_id']) & set(df.iloc[test_idx]['client_hash_id'])))

Train clients: 30
Test clients: 8
Overlap check (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 code cell
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Baseline (Week 4 method, but now on the grouped split for a fair comparison)
baseline = LinearRegression().fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_r2 = r2_score(y_test, baseline_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

# This week's model
rf = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)

import pandas as pd
comparison = pd.DataFrame({
    'Model': ['Linear Regression (baseline)', 'Random Forest'],
    'R2': [baseline_r2, rf_r2],
    'MAE': [baseline_mae, rf_mae]
})
print(comparison)

                          Model        R2        MAE
0  Linear Regression (baseline) -0.013452  11.506883
1                 Random Forest -0.007388  11.428573


Both models score negative R² on the grouped-by-client split — meaning neither beats simply predicting the average gsc_avg_position for a client the model has never seen. Random Forest is marginally less bad than the linear baseline (less negative R², slightly lower MAE), but "less bad" is not "good": with only 5 same-day behavioral features, this data does not meaningfully predict ranking position for a new client.

This is a real, honest regression from Week 4's result (R² ≈ 0.0033 on an ungrouped random split). The drop reveals that Week 4's small positive score was likely inflated by the model implicitly learning client-specific patterns from rows of the same clients appearing in both train and test — exactly the risk grouped validation is designed to catch. The grouped split is the more trustworthy number, even though it's a worse one.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 code cell
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'feature': features,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print(importance_df)

# Error analysis: where is the model most wrong?
results = X_test.copy()
results['actual'] = y_test
results['predicted'] = rf_pred
results['abs_error'] = (results['actual'] - results['predicted']).abs()
print("\nWorst 10 predictions:")
print(results.sort_values('abs_error', ascending=False).head(10))

print("\nError by impressions bucket:")
results['impressions_bucket'] = pd.cut(results['gsc_impressions'], bins=[-1,10,100,1000,1e9], labels=['0-10','11-100','101-1000','1000+'])
print(results.groupby('impressions_bucket')['abs_error'].mean())

                feature  importance_mean  importance_std
2      sessions_organic         0.025179        0.000460
0       gsc_impressions         0.011702        0.000427
4         scroll_events         0.001431        0.000126
3           sessions_ai         0.001339        0.000396
1  ga4_engaged_sessions         0.000018        0.000037

Worst 10 predictions:
         gsc_impressions  ga4_engaged_sessions  sessions_organic  sessions_ai  \
3601441                1                     0                 0            0   
2692214                1                     0                 0            0   
2692581                1                     0                 0            0   
2103245                1                     0                 0            0   
2212761                1                     0                 0            0   
2213138                1                     0                 0            0   
1293053                1                     0                 0    

/tmp/ipykernel_2161/2903501383.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(results.groupby('impressions_bucket')['abs_error'].mean())


What the model leans on: Permutation importance ranks sessions_organic highest (0.025), then gsc_impressions (0.012) — both far ahead of scroll_events, sessions_ai, and ga4_engaged_sessions (all near zero). But these importance values are uniformly small, consistent with the near-zero R²: the model isn't leaning heavily on anything, because nothing in this feature set carries much real signal for this target.

Where it's most wrong: The worst 10 predictions all share a pattern — 1-2 impressions, and an actual gsc_avg_position in the 200s-300s, while the model predicts around 15 (close to what looks like the training mean). With only 1-2 impressions, gsc_avg_position is an extremely noisy statistic — a single unusual query ranking very deep can swing the average wildly, and the model has no way to know this from same-day behavioral features alone. This shows up clearly in the error-by-impressions-bucket table: the 0-10 impressions bucket has the highest average error (14.6), while every higher-impression bucket sits lower and more consistent (~9.1-9.7).

What this means in practice: Low-impression content-days are inherently unpredictable with this feature set — the honest fix isn't a better model, it's excluding or separately handling very-low-impression rows, since their target values are dominated by noise rather than any pattern a model could learn.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.